# Fine-tune the small model, then test it

This notebook trains Gemma 3 4B on 800 generated requests, then tests it on the 150 unseen requests, alone and with the layers of code around it.

**Before you start**

1. In the menu choose **Runtime > Change runtime type**, pick **T4 GPU**, and save.
2. Choose **Runtime > Run all**.
3. Keep this tab open. It takes about two hours.

At the end a file named `finetune_results.zip` downloads. That file is the result.

Nothing here needs an account, a key or a payment.

## 1. Install and get the project

In [ ]:
%%capture
!pip install -q unsloth
!rm -rf /content/slm-opti && git clone -q --depth 1 https://github.com/eshwarpotturi/slm-opti /content/slm-opti

## 2. Load the model

In [ ]:
import os, json, time, glob, zipfile
os.chdir("/content/slm-opti")
import torch
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > T4 GPU, then Runtime > Run all."

from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-3-4b-it",   # the same Gemma 3 4B weights, hosted without a sign-in
    max_seq_length=4096,
    load_in_4bit=True,
    full_finetuning=False,
)
print("Model loaded.")

In [ ]:
import stack

def _msgs(messages):
    return [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in messages]

@torch.no_grad()
def hf_chat(messages, temperature=0, max_tokens=400):
    inputs = tokenizer.apply_chat_template(_msgs(messages), add_generation_prompt=True, tokenize=True,
                                           return_dict=True, return_tensors="pt").to("cuda")
    kw = dict(max_new_tokens=max_tokens, do_sample=temperature > 0)
    if temperature > 0:
        kw["temperature"] = temperature
    out = model.generate(**inputs, **kw)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

stack.HF_CHAT = hf_chat
print("Check, 2 + 2 =", hf_chat([{"role": "user", "content": "What is 2 + 2? Reply with the number only."}], max_tokens=5))

## 3. The untuned model on the unseen test

This is the fair starting point: the same model, loaded the same way, before any training. About 20 minutes.

In [ ]:
TEST = "tasks_third.json"
stack.main(["hf:gemma-3-4b-base", "--tasks", TEST, "--layers", "0", "--tag", "ft"])

## 4. Train

800 requests on accounts that appear in no test. Over half are reworded in four styles by a different AI model from the one that reworded the test. About 40 minutes.

In [ ]:
from datasets import Dataset

N_TRAIN = 800
tools_text = json.dumps(json.load(open("tools.json")), indent=1)
template = open("prompts/baseline.txt").read()

def prompt_of(t):
    return template.replace("{tools}", tools_text).replace("{account}", json.dumps(t["account"], indent=1)).replace("{request}", t["request"])

train = json.load(open("train_tasks.json"))[:N_TRAIN]
rows = []
for t in train:
    msgs = [{"role": "user", "content": [{"type": "text", "text": prompt_of(t)}]},
            {"role": "assistant", "content": [{"type": "text", "text": json.dumps(t["answer"])}]}]
    rows.append({"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False).removeprefix("<bos>")})
ds = Dataset.from_list(rows)
longest = max(len(tokenizer(r["text"]).input_ids) for r in rows[:60])
print(len(rows), "training examples; longest of the first 60 is", longest, "tokens")
assert longest < 4096, "A training example is longer than the model's window."
print(rows[0]["text"][-400:])

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407,
)

from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1, gradient_accumulation_steps=8,
        warmup_steps=5, num_train_epochs=1, learning_rate=2e-4,
        logging_steps=5, optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="linear",
        seed=3407, report_to="none", output_dir="outputs",
    ),
)
# learn only from the answer, not from the long prompt
trainer = train_on_responses_only(trainer, instruction_part="<start_of_turn>user\n", response_part="<start_of_turn>model\n")
t0 = time.time()
stats = trainer.train()
json.dump({"minutes": round((time.time() - t0) / 60, 1), "examples": len(rows), "log": trainer.state.log_history},
          open("results/finetune_training_log.json", "w"))
print("Training took %.0f minutes." % ((time.time() - t0) / 60))

## 5. The tuned model on the unseen test

Three runs: alone, with the rule check only, and with layers 1 to 4. About 50 minutes.

In [ ]:
model.eval()
print("Check, 2 + 2 =", hf_chat([{"role": "user", "content": "What is 2 + 2? Reply with the number only."}], max_tokens=5))
for layers in ["0", "4", "1234"]:
    stack.main(["hf:gemma-3-4b-tuned", "--tasks", TEST, "--layers", layers, "--tag", "ft"])

## 6. Results

In [ ]:
from score import call_ok
T = {t["id"]: t for t in json.load(open(TEST))}
WRITE = {"create_invoice", "send_invoice", "send_invoice_reminder", "cancel_sent_invoice", "record_payment_for_invoice",
         "create_refund", "accept_dispute_claim", "create_shipment_tracking", "cancel_subscription", "create_product"}
names = {"hf-gemma-3-4b-base__L0-ft": "Untuned, alone", "hf-gemma-3-4b-tuned__L0-ft": "Tuned, alone",
         "hf-gemma-3-4b-tuned__L4-ft": "Tuned + rule check", "hf-gemma-3-4b-tuned__L1234-ft": "Tuned + layers 1 to 4"}
print("%-26s %8s %14s" % ("Setup", "Right", "Wrong actions"))
for stem, label in names.items():
    f = "results/%s.jsonl" % stem
    if not os.path.exists(f):
        continue
    rs = list({json.loads(l)["id"]: json.loads(l) for l in open(f)}.values())
    wrong = sum(any(c["tool"] in WRITE and not any(call_ok(c, g) for g in T[r["id"]]["expected"]) for c in r["calls"]) for r in rs)
    print("%-26s %7.0f%% %14d   (%d requests)" % (label, 100.0 * sum(r["pass"] for r in rs) / len(rs), wrong, len(rs)))

with zipfile.ZipFile("/content/finetune_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("results/hf-*-ft.jsonl") + ["results/finetune_training_log.json"]:
        if os.path.exists(f):
            z.write(f)
from google.colab import files
files.download("/content/finetune_results.zip")